1. Criação do SCHEMA da camada silver

In [0]:
%sql

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS silver;

2. Definição do catálogo e SCHEMA


In [0]:
catalogo = "medalhao"
silver_db_name = "silver"

3. Importação das funções

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType, TimestampType

4. Função para deduplicação sênior

In [0]:
def deduplicar_por_ingestao(df, coluna_id, coluna_ingestao="timestamp_ingestion"):
    
    # Mantém apenas o registro mais recente para cada ID, utilizando Window Function ordenada pela coluna de ingestão em ordem decrescente.
    
    janela = Window.partitionBy(coluna_id).orderBy(F.col(coluna_ingestao).desc())
    
    return (
        df.withColumn("rn", F.row_number().over(janela))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

Transformações de limpeza e padronização das tabelas

1. silver.dim_consumidores (origem: tb_customers):

In [0]:
# Lê a tabela de clientes da camada Bronze
df_customers = spark.table(f"{catalogo}.bronze.tb_customers")

# Seleciona, renomeia, padroniza e tipa as colunas
df_consumidores = (
    df_customers
    .select(
        F.col("customer_id").cast(StringType()).alias("id_consumidor"),
        F.col("customer_unique_id").cast(StringType()).alias("id_unico_consumidor"),
        F.col("customer_zip_code_prefix").cast(IntegerType()).alias("prefixo_cep"),
        F.upper(F.col("customer_city")).cast(StringType()).alias("cidade"),
        F.upper(F.col("customer_state")).cast(StringType()).alias("estado"),
        F.col("customer_name").cast(StringType()).alias("nome_consumidor"),
        F.col("customer_gender").cast(StringType()).alias("genero_consumidor"),
        F.col("customer_birth_date").cast(DateType()).alias("data_nascimento"),
        F.col("customer_age").cast(IntegerType()).alias("idade_consumidor"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")   
    )
)

# Remove duplicidades mantendo o registro mais recente de cada consumidor
df_consumidores = deduplicar_por_ingestao(
    df_consumidores,
    "id_consumidor",
    "timestamp_ingestion"
)

# Grava na camada Silver
df_consumidores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_consumidores")

2. silver.fat_pedidos (origem: tb_orders):

In [0]:
# Lê a tabela de pedidos da camada Bronze
df_orders = spark.table(f"{catalogo}.bronze.tb_orders")

# Traduz os status dos pedidos do inglês para o português
status_traduzido = (
    F.when(F.col("order_status") == "delivered", "entregue")
     .when(F.col("order_status") == "canceled", "cancelado")
     .when(F.col("order_status") == "shipped", "enviado")
     .when(F.col("order_status") == "processing", "processando")
     .when(F.col("order_status") == "invoiced", "faturado")
     .when(F.col("order_status") == "unavailable", "indisponível")
     .when(F.col("order_status") == "created", "criado")
     .when(F.col("order_status") == "approved", "aprovado")

)

# Seleciona, renomeia, tipa e cria colunas derivadas
df_pedidos = (
    df_orders
    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("customer_id").cast(StringType()).alias("id_consumidor"),
        status_traduzido.alias("status"),
        F.to_timestamp("order_purchase_timestamp").alias("data_pedido"),
        F.to_timestamp("order_approved_at").alias("data_aprovacao"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_envio_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_entrega_real"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_entrega_estimada"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")
    )
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega_real"), F.col("data_pedido"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_pedido"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(F.col("status") != "entregue", "Não Entregue")
         .when(F.col("diferenca_entrega_dias") <= 0, "Sim")
         .otherwise("Não")
    )
)

# Grava na camada Silver
df_pedidos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedidos")

3. silver.fat_itens_pedidos (origem: tb_order_items):

In [0]:
# Lê a tabela de itens dos pedidos da camada Bronze
df_order_items = spark.table(f"{catalogo}.bronze.tb_order_items")

# Seleciona, renomeia e tipa as colunas
df_itens_pedidos = (
    df_order_items
    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("order_item_id").cast(IntegerType()).alias("id_item"),
        F.col("product_id").cast(StringType()).alias("id_produto"),
        F.col("seller_id").cast(StringType()).alias("id_vendedor"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion"),
        F.to_timestamp("shipping_limit_date").alias("data_limite_envio"),
        F.round(F.col("price").cast(DoubleType()), 2).alias("preco_BRL"),
        F.round(F.col("freight_value").cast(DoubleType()), 2).alias("preco_frete")
    )
)

# Grava na camada Silver
df_itens_pedidos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_itens_pedidos")

4. silver.fat_pagamentos_pedidos (origem: tb_order_payments):

In [0]:
# Lê a tabela de pagamentos da camada Bronze
df_order_payments = spark.table(f"{catalogo}.bronze.tb_order_payments")

# Traduz os tipos de pagamento para o português
tipo_pagamento_traduzido = (
    F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
     .when(F.col("payment_type") == "boleto", "Boleto")
     .when(F.col("payment_type") == "voucher", "Voucher")
     .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
     .when(F.col("payment_type") == "not_defined", "Não Definido")
)

# Seleciona, renomeia e tipa as colunas
df_pagamentos = (
    df_order_payments
    .select(
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.col("payment_sequential").cast(IntegerType()).alias("sequencial_pagamento"),
        tipo_pagamento_traduzido.alias("tipo_pagamento"),
        F.col("payment_installments").cast(IntegerType()).alias("quantidade_parcelas"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion"),
        F.round(F.col("payment_value").cast(DoubleType()), 2).alias("valor_pagamento_BRL")

    )
)

# Grava na camada Silver
df_pagamentos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")

5. silver.fat_avaliacoes_pedidos (origem: tb_order_reviews):

In [0]:

# Lê a tabela de avaliações da camada Bronze
df_order_reviews = spark.table(f"{catalogo}.bronze.tb_order_reviews")

# Lê os pedidos válidos da Silver para eliminar avaliações com pedidos inexistentes
df_pedidos_validos = (
    spark.table(f"{catalogo}.{silver_db_name}.fat_pedidos")
    .select("id_pedido")
    .distinct()
)

# Aplica tratamento de datas com try_to_timestamp, trata nulos e remove registros inválidos
df_avaliacoes = (
    df_order_reviews
    .select(
        F.col("review_id").cast(StringType()).alias("id_avaliacao"),
        F.col("order_id").cast(StringType()).alias("id_pedido"),
        F.expr("try_cast(review_score as int)").alias("nota_avaliacao"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion"),
        F.coalesce(F.col("review_comment_title"), F.lit("Sem título")).alias("titulo_avaliacao_comentario"),
        F.coalesce(F.col("review_comment_message"), F.lit("Sem comentário")).alias("mensagem_avaliacao_comentario"),
        F.expr("try_to_timestamp(review_creation_date)").alias("data_criacao_avaliacao"),
        F.expr("try_to_timestamp(review_answer_timestamp)").alias("data_resposta_avaliacao")
    )
    .withColumn(
        "titulo_avaliacao_comentario",
        F.when(F.trim(F.col("titulo_avaliacao_comentario")) == "", "Sem título")
         .otherwise(F.col("titulo_avaliacao_comentario"))
    )
    .withColumn(
        "mensagem_avaliacao_comentario",
        F.when(F.trim(F.col("mensagem_avaliacao_comentario")) == "", "Sem comentário")
         .otherwise(F.col("mensagem_avaliacao_comentario"))
    )
    .join(df_pedidos_validos, on="id_pedido", how="inner")
    .filter(F.col("nota_avaliacao").isNotNull())
    .filter((F.col("nota_avaliacao") >= 1) & (F.col("nota_avaliacao") <= 5))
    .filter(
        (F.col("data_criacao_avaliacao").isNull()) |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
    .filter(
        (F.col("data_resposta_avaliacao").isNull()) |
        (F.col("data_resposta_avaliacao") <= F.current_timestamp())
    )
)

# Grava na camada Silver
df_avaliacoes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos")

6. silver.dim_produtos (origem: tb_products):

In [0]:
# Lê a tabela de produtos da camada Bronze
df_products = spark.table(f"{catalogo}.bronze.tb_products")

# Seleciona, renomeia, tipa e mantém a coluna de ingestão
df_produtos = (
    df_products
    .select(
        F.col("product_id").cast(StringType()).alias("id_produto"),
        F.col("product_name").cast(StringType()).alias("nome_produto"),
        F.col("product_category_name").cast(StringType()).alias("categoria_produto"),
        F.col("product_weight_g").cast(DoubleType()).alias("peso_produto_gramas"),
        F.col("product_length_cm").cast(DoubleType()).alias("comprimento_centimetros"),
        F.col("product_height_cm").cast(DoubleType()).alias("altura_centimetros"),
        F.col("product_width_cm").cast(DoubleType()).alias("largura_centimetros"),
        F.col("product_photos_qty").cast(IntegerType()).alias("quantidade_fotos"),
        F.col("product_name_lenght").cast(IntegerType()).alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast(IntegerType()).alias("tamanho_descricao_produto"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")
    )
)

# Remove duplicidades mantendo o registro mais recente de cada produto
df_produtos = deduplicar_por_ingestao(
    df_produtos,
    "id_produto",
    "timestamp_ingestion"
)

# Grava na camada Silver
df_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_produtos")

7. silver.dim_vendedores (origem: tb_sellers):

In [0]:
# Lê a tabela de vendedores da camada Bronze
df_sellers = spark.table(f"{catalogo}.bronze.tb_sellers")

# Seleciona, renomeia, padroniza e tipa as colunas
df_vendedores = (
    df_sellers
    .select(
        F.col("seller_id").cast(StringType()).alias("id_vendedor"),
        F.col("seller_name").cast(StringType()).alias("nome_vendedor"),
        F.col("seller_zip_code_prefix").cast(StringType()).alias("prefixo_cep"),
        F.upper(F.col("seller_city")).cast(StringType()).alias("cidade"),
        F.upper(F.col("seller_state")).cast(StringType()).alias("estado"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion"),
        F.col("seller_registration_date").cast(DateType()).alias("data_registro_vendedor")
    )
)

# Remove duplicidades mantendo o registro mais recente de cada vendedor
df_vendedores = deduplicar_por_ingestao(
    df_vendedores,
    "id_vendedor",
    "timestamp_ingestion"
)

# Grava na camada Silver
df_vendedores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_vendedores")

8. silver.dim_categoria_produtos_traducao (origem: tb_product_category_name_translation):

In [0]:
# Lê a tabela de tradução de categorias da camada Bronze
df_category_translation = spark.table(f"{catalogo}.bronze.tb_product_category_name_translation")

# Seleciona, renomeia e tipa as colunas
df_categoria_traducao = (
    df_category_translation
    .select(
        F.col("product_category_name").cast(StringType()).alias("nome_produto_pt"),
        F.col("product_category_name_english").cast(StringType()).alias("nome_produto_en"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")
    )
)

# Grava na camada Silver
df_categoria_traducao.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_categoria_produtos_traducao")

9. silver.dim_cotacao_dolar (origem: tb_cotacao_dolar):

In [0]:
# Lê a tabela de cotação do dólar da Bronze
df_cotacao_dolar = spark.table(f"{catalogo}.bronze.tb_cotacao_dolar")

# Trata e tipa as colunas
df_cotacao_dolar_tratada = (
    df_cotacao_dolar
    .select(
        F.col("cotacaoCompra").cast(DoubleType()).alias("cotacao_dolar_compra"),
        F.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
        F.to_date("dataHoraCotacao").alias("data_referencia"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")
    )
)

# Mantém apenas uma cotação por dia, escolhendo a mais recente
janela_dia = Window.partitionBy("data_referencia").orderBy(F.col("data_hora_cotacao").desc())

df_cotacao_por_dia = (
    df_cotacao_dolar_tratada
    .withColumn("rn", F.row_number().over(janela_dia))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Descobre intervalo de datas
datas = df_cotacao_por_dia.select(
    F.min("data_referencia").alias("data_min"),
    F.max("data_referencia").alias("data_max")
).collect()[0]

data_min = datas["data_min"]
data_max = datas["data_max"]

# Cria calendário contínuo
df_calendario = spark.sql(f"""
SELECT explode(
    sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)
) AS data_referencia
""")

# Junta calendário com cotações reais
df_cotacao_calendario = df_calendario.join(
    df_cotacao_por_dia,
    on="data_referencia",
    how="left"
)

# Janela para preenchimento
janela_preenchimento = (
    Window.orderBy("data_referencia")
    .rowsBetween(Window.unboundedPreceding, 0)
)

# Preenche faltas com última cotação válida anterior
df_dim_cotacao_dolar = (
    df_cotacao_calendario
    .withColumn(
        "cotacao_dolar_compra",
        F.last("cotacao_dolar_compra", ignorenulls=True).over(janela_preenchimento)
    )
    .withColumn(
        "data_hora_cotacao",
        F.last("data_hora_cotacao", ignorenulls=True).over(janela_preenchimento)
    )
    .withColumn(
        "timestamp_ingestion",
        F.last("timestamp_ingestion", ignorenulls=True).over(janela_preenchimento)
    )
    .select(
        F.col("data_referencia").cast(DateType()).alias("data_referencia"),
        F.round(F.col("cotacao_dolar_compra"), 2).alias("cotacao_dolar_compra"),
        F.col("data_hora_cotacao").cast(TimestampType()).alias("data_hora_cotacao"),
        F.col("timestamp_ingestion").cast(TimestampType()).alias("timestamp_ingestion")
    )
)

# Grava na Silver
df_dim_cotacao_dolar.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


10. Tabela Final Silver (silver.fat_pedido_total):

In [0]:
# Lê as tabelas da Silver
df_pedidos = spark.table(f"{catalogo}.{silver_db_name}.fat_pedidos")
df_pagamentos = spark.table(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos")
df_cotacao_dolar = spark.table(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar")

# Soma os pagamentos de cada pedido para obter o valor total pago em BRL
df_pagamentos_agregado = (
    df_pagamentos
    .groupBy("id_pedido")
    .agg(
        F.round(F.sum("valor_pagamento_BRL"), 2).alias("valor_total_pago_brl")
    )
)

# Junta pedidos com pagamentos agregados e com a cotação do dólar
df_fat_pedido_total = (
    df_pedidos.alias("p")
    .join(
        df_pagamentos_agregado.alias("pg"),
        on="id_pedido",
        how="left"
    )
    .join(
        df_cotacao_dolar.alias("d"),
        F.to_date(F.col("p.data_pedido")) == F.col("d.data_referencia"),
        how="left"
    )
    .select(
        F.col("p.id_pedido").alias("id_pedido"),
        F.col("p.id_consumidor").alias("id_consumidor"),
        F.col("p.status").alias("status"),
        F.round(F.col("pg.valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        F.round(
            F.col("pg.valor_total_pago_brl") / F.col("d.cotacao_dolar_compra"),
            2
        ).alias("valor_total_pago_usd"),
        F.col("p.data_pedido").alias("data_pedido")
    )
)

# Grava na Silver
df_fat_pedido_total.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedido_total")

11. Otimização Física (Delta Optimize):

In [0]:
# Otimização física das principais tabelas fato da camada Silver

spark.sql(f"""
OPTIMIZE {catalogo}.{silver_db_name}.fat_pedidos
ZORDER BY (id_pedido, data_pedido)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{silver_db_name}.fat_pedido_total
ZORDER BY (id_pedido, data_pedido)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{silver_db_name}.fat_avaliacoes_pedidos
ZORDER BY (id_pedido, data_criacao_avaliacao)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{silver_db_name}.fat_itens_pedidos
ZORDER BY (id_pedido, data_limite_envio)
""")

spark.sql(f"""
OPTIMIZE {catalogo}.{silver_db_name}.fat_pagamentos_pedidos
ZORDER BY (id_pedido)
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,